In [2]:
import pandas as pd
import re

# --- CONFIGURATION ---
LOG_FILE = '/Users/varattsaengsiripongpun/Documents/Reinforcement/HydraAIops/data/raw/HDFS_2k.log' 

print(f"Loading {LOG_FILE}...")

# 1. Read the file
with open(LOG_FILE, 'r') as f:
    lines = f.readlines()

print(f"Loaded {len(lines)} log lines.")
print("-" * 30)

# 2. Inspect the first 5 lines
print("--- Raw Log Sample ---")
for i, line in enumerate(lines[:5]):
    print(f"{i}: {line.strip()}")

print("-" * 30)

# 3. Basic Parsing (Regex)
# This extracts the distinct parts: Date, Time, PID, Level, and the Message
log_pattern = re.compile(r'(\d{6})\s+(\d{6})\s+(\d+)\s+(\w+)\s+(.*)')

data = []
for line in lines:
    match = log_pattern.match(line)
    if match:
        data.append({
            'date': match.group(1),
            'time': match.group(2),
            'pid': match.group(3),
            'level': match.group(4),
            'content': match.group(5) # The actual message we need to analyze
        })

# Convert to DataFrame
df_logs = pd.DataFrame(data)

print("--- Parsed Dataframe ---")
display(df_logs.head())

# Check if we have different log levels
print("\n--- Log Levels Distribution ---")
print(df_logs['level'].value_counts())

Loading /Users/varattsaengsiripongpun/Documents/Reinforcement/HydraAIops/data/raw/HDFS_2k.log...
Loaded 2000 log lines.
------------------------------
--- Raw Log Sample ---
0: 081109 203615 148 INFO dfs.DataNode$PacketResponder: PacketResponder 1 for block blk_38865049064139660 terminating
1: 081109 203807 222 INFO dfs.DataNode$PacketResponder: PacketResponder 0 for block blk_-6952295868487656571 terminating
2: 081109 204005 35 INFO dfs.FSNamesystem: BLOCK* NameSystem.addStoredBlock: blockMap updated: 10.251.73.220:50010 is added to blk_7128370237687728475 size 67108864
3: 081109 204015 308 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_8229193803249955061 terminating
4: 081109 204106 329 INFO dfs.DataNode$PacketResponder: PacketResponder 2 for block blk_-6670958622368987959 terminating
------------------------------
--- Parsed Dataframe ---


,date,time,pid,level,content
0,081109,203615,148,INFO,dfs.DataNode$PacketResponder: PacketResponder ...
1,081109,203807,222,INFO,dfs.DataNode$PacketResponder: PacketResponder ...
2,081109,204005,35,INFO,dfs.FSNamesystem: BLOCK* NameSystem.addStoredB...
3,081109,204015,308,INFO,dfs.DataNode$PacketResponder: PacketResponder ...
4,081109,204106,329,INFO,dfs.DataNode$PacketResponder: PacketResponder ...



--- Log Levels Distribution ---
level
INFO    1920
WARN      80
Name: count, dtype: int64


In [3]:
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.ensemble import IsolationForest
import numpy as np

# --- 1. NORMALIZATION ---
# We replace specific numbers/IPs with generic tokens so the AI sees the "pattern"
def normalize_log(text):
    # Regex to replace 'blk_12345' with 'BLK_ID'
    text = re.sub(r'blk_-?\d+', 'BLK_ID', text)
    # Regex to replace IP addresses (10.251...) with 'IP_ADDR'
    text = re.sub(r'\d+\.\d+\.\d+\.\d+(:\d+)?', 'IP_ADDR', text)
    # Regex to replace any other numbers with 'NUM'
    text = re.sub(r'\d+', 'NUM', text)
    return text

print("Normalizing logs...")
df_logs['log_template'] = df_logs['content'].apply(normalize_log)

# Show the difference
print("\n--- Before vs After ---")
display(df_logs[['content', 'log_template']].head())

# --- 2. VECTORIZATION ---
# Convert the text templates into a matrix of numbers
# max_features=50 means we only care about the top 50 most common words (sufficient for HDFS)
vectorizer = TfidfVectorizer(max_features=50)
X_logs = vectorizer.fit_transform(df_logs['log_template'])

print(f"\nVectorized Shape: {X_logs.shape} (Rows, Features)")

Normalizing logs...

--- Before vs After ---


,content,log_template
0,dfs.DataNode$PacketResponder: PacketResponder ...,dfs.DataNode$PacketResponder: PacketResponder ...
1,dfs.DataNode$PacketResponder: PacketResponder ...,dfs.DataNode$PacketResponder: PacketResponder ...
2,dfs.FSNamesystem: BLOCK* NameSystem.addStoredB...,dfs.FSNamesystem: BLOCK* NameSystem.addStoredB...
3,dfs.DataNode$PacketResponder: PacketResponder ...,dfs.DataNode$PacketResponder: PacketResponder ...
4,dfs.DataNode$PacketResponder: PacketResponder ...,dfs.DataNode$PacketResponder: PacketResponder ...



Vectorized Shape: (2000, 50) (Rows, Features)


In [4]:
# --- 3. ANOMALY DETECTION ---
# contamination=0.02 means we expect roughly 2% of logs to be anomalies
iso_model = IsolationForest(contamination=0.02, random_state=42)

print("Training Isolation Forest on logs...")
# -1 = Anomaly, 1 = Normal
df_logs['anomaly_score'] = iso_model.fit_predict(X_logs)

# --- 4. INSPECT RESULTS ---
# Let's see what the model thinks is "Weird"
anomalies = df_logs[df_logs['anomaly_score'] == -1]

print(f"\nFound {len(anomalies)} anomalies out of {len(df_logs)} logs.")

if not anomalies.empty:
    print("\n--- Top 5 Anomalous Logs ---")
    # Show the raw content to see what looks weird
    display(anomalies[['date','time', 'level', 'content', 'log_template']].head())
    
    print("\n--- Distribution of Anomalous Templates ---")
    print(anomalies['log_template'].value_counts().head())
else:
    print("No anomalies found. Try increasing 'contamination' parameter.")

Training Isolation Forest on logs...

Found 15 anomalies out of 2000 logs.

--- Top 5 Anomalous Logs ---


KeyError: "['datetime'] not in index"